In [0]:
import json, urllib.request
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql import functions as F, Window

latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"
LANDING_DIR = "/Volumes/workspace/spb/landing"
MIN_CONFIDENCE = 0.4
H3_RES = 8

def flatten_places(df):
    """Pull fields out of the taxonomy and bbox structs."""
    return (
        df.withColumn("lon", F.col("bbox.xmin"))
          .withColumn("lat", F.col("bbox.ymin"))
          .drop("taxonomy", "bbox")
    )

def flatten_buildings(df):
    return (
          df.withColumn("lon", (F.col("bbox.xmin") + F.col("bbox.xmax")) / 2)
            .withColumn("lat", (F.col("bbox.ymin") + F.col("bbox.ymax")) / 2)
            .drop("bbox")
    )

def filter_to_bar(df):
    return df.where(
        (F.col("confidence") >= MIN_CONFIDENCE) &
        (F.col("basic_category") == "bar"))
    
def filter_places(df):
    return df.where(
        (F.col("confidence") >= MIN_CONFIDENCE))

def dedupe(df):
    return df.dropDuplicates(["name"])


def add_h3(df):
    """Built-in H3 index. Note the argument order: longitude, then latitude."""
    return df.withColumn("h3_cell", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES})"))

In [0]:
bronze_places = spark.table("workspace.spb.bronze_places")
flat_places = flatten_places(bronze_places)
kept_places = filter_places(flat_places)
deduped_places = dedupe(kept_places)
silver_places = add_h3(deduped_places)

bars = filter_to_bar(flat_places)
dedupe_bar = dedupe(bars)
silver_bars = add_h3(dedupe_bar)

print("bronze", bronze_places.count())
print("after filter", kept_places.count())
print("after dedupe", deduped_places.count())


silver_places.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_places")
print("silver places", spark.table('workspace.spb.silver_places').count())

silver_bars.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_bars")
print("silver bars", spark.table('workspace.spb.silver_bars').count())

In [0]:
bronze_buildings = spark.table("workspace.spb.bronze_buildings")
flat_buldings = flatten_buildings(bronze_buildings)
deduped_buildings = dedupe(flat_buldings)
silver_buildings = add_h3(deduped_buildings)

silver_buildings.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_buildings")
print("silver buildings", spark.table("workspace.spb.silver_buildings").count())

In [0]:
nodes_b = spark.table("workspace.spb.bronze_walk_nodes")
edges_b = spark.table("workspace.spb.bronze_walk_edges")

silver_nodes = nodes_b.withColumn(
    "h3_cell", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES})")   # lon first
)
silver_nodes.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_walk_nodes")

In [0]:
K = 1
bars = spark.table("workspace.spb.silver_bars").select("id", "lon", "lat", "h3_cell")
nodes = (spark.table("workspace.spb.silver_walk_nodes")
         .select("node_id", F.col("lon").alias("n_lon"),
                 F.col("lat").alias("n_lat"), F.col("h3_cell").alias("n_cell")))

bars_cells = bars.withColumn("search_cell", F.explode(F.expr(f"h3_kring(h3_cell, {K})")))

In [0]:
cand = bars_cells.join(nodes, bars_cells.search_cell == nodes.n_cell)

R = 6371000.0   # earth radius in meters

def haversine_m(lat1, lon1, lat2, lon2):
    p1, p2 = F.radians(lat1), F.radians(lat2)
    a = (F.sin((p2 - p1) / 2) ** 2
         + F.cos(p1) * F.cos(p2) * F.sin(F.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R * F.asin(F.sqrt(a))

cand = cand.withColumn("snap_m", haversine_m(F.col("lat"), F.col("lon"),
                                             F.col("n_lat"), F.col("n_lon")))

In [0]:
w = Window.partitionBy("id").orderBy("snap_m")
snapped = (cand.withColumn("rn", F.row_number().over(w))
               .where("rn = 1")
               .select(F.col("id").alias("bar_id"), "node_id", "snap_m"))

snapped.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.snapped_walk_nodes")

In [0]:
w = Window.orderBy("node_id")
(spark.table("workspace.spb.silver_walk_nodes")
    .withColumn("node_idx", F.row_number().over(w) - 1)
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.spb.silver_walk_nodes"))

silver_nodes = spark.table("workspace.spb.silver_walk_nodes")   # re-read so the variable has node_idx
print(silver_nodes.columns)

In [0]:
idx = silver_nodes.select("node_id", "node_idx")
silver_edges = (edges_b.where("src <> dst")
    .join(idx.withColumnRenamed("node_id", "src").withColumnRenamed("node_idx", "src_idx"), "src")
    .join(idx.withColumnRenamed("node_id", "dst").withColumnRenamed("node_idx", "dst_idx"), "dst")
    .select("src", "dst", "src_idx", "dst_idx", "length_m"))
silver_edges.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_walk_edges")